<a href="https://colab.research.google.com/github/oluwayemidara1/portfolio/blob/main/comfyui_ltx25main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU - change runtime type!")

INSTALL_LTX_NODE = False   # the built-in templates don't need it; skipping makes the UI lighter

%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q "huggingface_hub>=1.32,<2.0"

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

if INSTALL_LTX_NODE:
    %cd /content/ComfyUI/custom_nodes
    !git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git
    !pip install -q -r ComfyUI-LTXVideo/requirements.txt
    %cd /content/ComfyUI

NVIDIA L4, 23034 MiB

/content
Cloning into 'ComfyUI'...
remote: Enumerating objects: 61599, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 61599 (delta 168), reused 90 (delta 87), pack-reused 61374 (from 4)
Receiving objects: 100% (61599/61599), 97.15 MiB | 20.00 MiB/s, done.
Resolving deltas: 100% (42614/42614), done.
/content/ComfyUI


In [2]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # Colab Secret, notebook access ON

VARIANT = "int8"   # "int8" for L4 (24GB); "bf16" for A100/H100

files = [
    "vae/ltx-2.5-video-vae-bf16.safetensors",
    "vae/ltx-2.5-audio-vae-bf16.safetensors",
    "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
    "model_patches/ltx-2.5-duration-head-bf16.safetensors",
]
if VARIANT == "int8":
    files += [
        "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
        "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors",
    ]
else:
    files += [
        "diffusion_models/ltx-2.5-22b-distilled-transformer-bf16.safetensors",
        "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors",
    ]

!hf download Lightricks/LTX-2.5 {" ".join(files)} --local-dir /content/ComfyUI/models

# Small prompt-enhancer encoder that the LTX 2.5 template asks for
!hf download Comfy-Org/gemma-4 text_encoders/gemma4_e2b_it_int8_convrot.safetensors --local-dir /content/ComfyUI/models

!find /content/ComfyUI/models -name "*.safetensors" -exec ls -lh {} \;

Hint: A new version of huggingface_hub (2.3.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0% 0/6 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/15.4G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/17.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/18.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/39.7G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/39.7G [00:00<?, ?B/s]

Reconstructing (incomplete total...):   1% 246M/39.7G [00:02<03:21, 196MB/s, 24.5MB/s  ]  


Reconstructing (incomplete total...):   9% 3.62G/39.7G [00:11<02:12, 273MB/s,  815MB/s  ]
Reconstructing (incomplete total...):  10% 3.94G/39.7G [00:11<01:28, 406MB/s,  273MB/s  ]
Reconstruct

In [ ]:
import subprocess, time, re, socket

ACCESS = "colab"   # "colab" = Google proxy link, "cloudflare" = trycloudflare link

!pkill -f "main.py"; pkill -f cloudflared; true
time.sleep(2)

log = open("/content/comfy.log", "w")
comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)

def port_open():
    with socket.socket() as s:
        s.settimeout(1)
        return s.connect_ex(("127.0.0.1", 8188)) == 0

def tail():
    print("".join(open("/content/comfy.log").readlines()[-40:]))

for i in range(150):
    if comfy.poll() is not None:
        print("ComfyUI crashed. Last log lines:\n"); tail(); raise SystemExit
    if port_open():
        print(f"ComfyUI is up after ~{i*2}s"); break
    time.sleep(2)
else:
    print("Timed out. Last log lines:\n"); tail(); raise SystemExit

if ACCESS == "colab":
    from google.colab import output
    output.serve_kernel_port_as_window(8188)   # click the link it prints
else:
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--protocol", "http2", "--url", "http://127.0.0.1:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tunnel.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("Open ComfyUI here:", m.group(0)); break

while comfy.poll() is None:
    time.sleep(30)
print("ComfyUI stopped. Last log lines:\n"); tail()

^C
ComfyUI is up after ~26s
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>